
# 練習問題: パーコレーション (浸透) と相転移

## 目標

L×L の格子の各マスを, 確率 `p` で「通れる (開)」, 確率 `1−p` で「通れない」にする。上端から下端まで, 開マスを上下左右にたどって到達できるとき「**浸透した**」と呼ぶ。多数の試行で**浸透する確率**を推定し, `p` を変えると**ある臨界値で確率が急変する (相転移)** 様子を観察する。

独立な試行を `parallel for` で分担し, 浸透した回数を `reduction` で数える, というモンテカルロの並列パターンを用いる。各試行は探索量が違う (浸透すると途中で打ち切る) ので `schedule(dynamic)` が効く。

## 課題

`percolation.cpp` (または `percolation.f90`) は, 1試行 (格子を作り, 上端から下端へ到達できるか深さ優先探索で判定) を行う関数 `one_trial` を持ち, それを `T` 回繰り返して浸透確率を求める。各試行は独立なので並列化できる。

`TODO` の箇所に **OpenMP の指示を追加** し, 試行ループを並列化して浸透回数 `perc` を集計せよ。

- C++: ループの直前に `#pragma omp parallel for reduction(+:perc) schedule(dynamic)` を1行加える。
- Fortran: `do` ループを `!$omp parallel do reduction(+:perc) schedule(dynamic)` と `!$omp end parallel do` で囲む。

セルの開閉は, 状態を持たない乱数 `draw_rand01(seed=試行番号, k=セル番号)` で決めている。どのスレッドが計算しても格子は同じになるので, **浸透確率はスレッド数によらず一致**する (浸透回数は整数なので `reduction` で完全に同じ値になる)。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore percolation.cpp -o percolation.exe

# Fortran
nvfortran -fast -mp=multicore percolation.f90 -o percolation.exe
```

第1引数で格子サイズ `L`, 第2引数で確率 `p`, 第3引数で試行回数 `T`。

```
OMP_NUM_THREADS=4 ./percolation.exe 128 0.6 2000
```

`p` を変えながら一気に調べると相転移が見える:

```
for p in 0.50 0.55 0.58 0.59 0.60 0.62 0.65 0.70 ; do
    OMP_NUM_THREADS=4 ./percolation.exe 128 $p 2000
done
```

## 期待される結果

正方格子のサイト・パーコレーションの臨界確率は約 `p_c ≈ 0.5927`。

- `p` が `p_c` より十分小さいと, ほとんど浸透しない (確率 ≈ 0)。
- `p` が `p_c` より十分大きいと, ほぼ必ず浸透する (確率 ≈ 1)。
- `p_c` 付近で確率が 0 から 1 へ急に立ち上がる (**相転移**)。`p = 0.5927` でちょうど浸透確率が約 0.5 になる。

```
L=128, p=0.500: 浸透確率 ≈ 0.00
L=128, p=0.593: 浸透確率 ≈ 0.52
L=128, p=0.650: 浸透確率 ≈ 1.00
```

- 格子サイズ `L` を大きくすると, 立ち上がりが `p_c` 付近でより鋭くなる (理想的には階段関数に近づく) ことも確かめよ。
- `OMP_NUM_THREADS` を変えても浸透確率が変わらないこと (並列化が正しいこと) も確認せよ。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ percolation.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* 状態を持たない (カウンタベースの) 乱数: (seed,k) から [0,1) の値を決める純粋関数。
   セル k の開閉を draw_rand01(seed=試行番号, k=セル番号) で決めるので, どのスレッドが
   担当しても同じ格子になり, スレッド数によらず結果が一致する (共有状態なし=競合なし)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;   /* 2^31 - 1 */
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;        /* [0,1) */
}

/* 1試行: L×L の各セルを確率 p で「開」にした格子で,
   上端の開セルから下端の開セルへ (上下左右の開セル伝いに) たどり着けるか。
   たどり着ければ 1 (浸透した), さもなくば 0 を返す。深さ優先探索 (スタック) で判定。 */
int one_trial(int L, double p, long long seed) {
  int n = L * L;
  unsigned char * open = (unsigned char *)malloc(n);
  unsigned char * vis  = (unsigned char *)calloc(n, 1);
  int *           stk  = (int *)malloc(sizeof(int) * n);
  for (int i = 0; i < n; i++) open[i] = (draw_rand01(seed, i) < p) ? 1 : 0;

  int sp = 0;
  for (int c = 0; c < L; c++) {               /* 上端 (行0) の開セルを出発点に */
    if (open[c]) { vis[c] = 1; stk[sp++] = c; }
  }
  int perc = 0;
  const int dr[4] = { -1, 1, 0, 0 };
  const int dc[4] = { 0, 0, -1, 1 };
  while (sp > 0) {
    int idx = stk[--sp];
    int r = idx / L, c = idx % L;
    if (r == L - 1) { perc = 1; break; }      /* 下端に到達 = 浸透 */
    for (int d = 0; d < 4; d++) {
      int nr = r + dr[d], nc = c + dc[d];
      if (nr < 0 || nr >= L || nc < 0 || nc >= L) continue;
      int nidx = nr * L + nc;
      if (open[nidx] && !vis[nidx]) { vis[nidx] = 1; stk[sp++] = nidx; }
    }
  }
  free(open); free(vis); free(stk);
  return perc;
}

int main(int argc, char ** argv) {
  int    L = (argc > 1 ? atoi(argv[1]) : 128);     /* 格子の一辺 */
  double p = (argc > 2 ? atof(argv[2]) : 0.6);     /* セルが開く確率 */
  long   T = (argc > 3 ? atol(argv[3]) : 2000);    /* 試行回数 */
  long   perc = 0;

  /* T 回の試行は互いに独立。浸透した回数を数える。
     試行ごとに探索量が違う (浸透すると途中で打ち切る) ので schedule(dynamic) が有効。 */
  double t0 = omp_get_wtime();
  // TODO: 各試行は独立。#pragma omp parallel for reduction(+:perc) schedule(dynamic) で並列化・集計せよ.
  for (long t = 0; t < T; t++) {
    perc += one_trial(L, p, t);
  }
  double elapsed = omp_get_wtime() - t0;
  printf("L=%d, p=%.3f, trials=%ld: 浸透確率 = %.4f\n", L, p, T, (double)perc / T);
  printf("elapsed = %.3f sec\n", elapsed);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore percolation.cpp -o percolation_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./percolation_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ percolation.f90
module percolation_mod
contains
  ! 状態を持たない (カウンタベースの) 乱数: (seed,k) から [0,1) の値を決める純粋関数。
  ! セル k の開閉を draw_rand01(seed=試行番号, k=セル番号) で決めるので, どのスレッドが
  ! 担当しても同じ格子になり, スレッド数によらず結果が一致する (共有状態なし=競合なし)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8   ! 2^31 - 1
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)        ! [0,1)
  end function draw_rand01

  ! 1試行: L×L の各セルを確率 p で「開」にした格子で, 上端の開セルから
  ! 下端の開セルへたどり着けるか。たどり着ければ 1, さもなくば 0。深さ優先探索で判定。
  ! セル番号は 0..L*L-1 (idx = r*L + c, r,c は 0 始まり)。
  function one_trial(L, p, seed) result(perc)
    integer, intent(in) :: L
    real(8), intent(in) :: p
    integer(8), intent(in) :: seed
    integer :: perc
    integer :: n, i, c, r, sp, idx, d, nr, nc, nidx
    logical, allocatable :: open(:), vis(:)
    integer, allocatable :: stk(:)
    integer :: dr(4), dc(4)
    dr = (/ -1, 1, 0, 0 /)
    dc = (/ 0, 0, -1, 1 /)
    n = L * L
    allocate(open(0:n-1), vis(0:n-1), stk(0:n-1))
    do i = 0, n - 1
       open(i) = (draw_rand01(seed, int(i, 8)) < p)
    end do
    vis = .false.
    sp = 0
    do c = 0, L - 1                 ! 上端 (行0) の開セルを出発点に
       if (open(c)) then
          vis(c) = .true.; stk(sp) = c; sp = sp + 1
       end if
    end do
    perc = 0
    do while (sp > 0)
       sp = sp - 1; idx = stk(sp)
       r = idx / L; c = mod(idx, L)
       if (r == L - 1) then
          perc = 1; exit           ! 下端に到達 = 浸透
       end if
       do d = 1, 4
          nr = r + dr(d); nc = c + dc(d)
          if (nr < 0 .or. nr >= L .or. nc < 0 .or. nc >= L) cycle
          nidx = nr * L + nc
          if (open(nidx) .and. .not. vis(nidx)) then
             vis(nidx) = .true.; stk(sp) = nidx; sp = sp + 1
          end if
       end do
    end do
    deallocate(open, vis, stk)
  end function one_trial
end module percolation_mod

program percolation
  use percolation_mod
  use omp_lib
  character(len=32) :: arg
  integer :: L
  real(8) :: p, t0, elapsed
  integer(8) :: T, t_, perc
  L = 128
  p = 0.6d0
  T = 2000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) L
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) p
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) T
  end if
  perc = 0_8

  ! T 回の試行は互いに独立。浸透した回数を数える。
  ! 試行ごとに探索量が違うので schedule(dynamic) が有効。
  t0 = omp_get_wtime()
  ! TODO: 各試行は独立。!$omp parallel do reduction(+:perc) schedule(dynamic) で並列化・集計せよ.
  do t_ = 0, T - 1
     perc = perc + one_trial(L, p, t_)
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  elapsed = omp_get_wtime() - t0

  print "(a,i0,a,f0.3,a,i0,a,f0.4)", &
       "L=", L, ", p=", p, ", trials=", T, ": 浸透確率 = ", real(perc, 8) / T
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program percolation

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore percolation.f90 -o percolation_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./percolation_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:percolation.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:percolation.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:percolation.cpp}